# Module 21 — Data Pipelines for Large Corpora

Every module so far loaded its (tiny) corpus entirely into a Python string
or list. Module 30's real pretraining corpus will be gigabytes of text —
far too big to hold as one in-memory blob comfortably, and documents come
in wildly different lengths. This module covers the two practical
techniques that matter: **concatenate-and-chunk** (instead of padding each
document individually) and **streaming** (processing documents one at a
time instead of materializing the whole corpus at once).

## 1. The naive approach: pad every document to the same length

Simple, but wasteful — short documents burn a lot of their batch slot on
meaningless padding tokens the model has to explicitly learn to ignore.

In [ ]:
import torch

documents = [
    "aether and lumine are twins known as the traveler",
    "paimon is always hungry",
    "klee loves to explore the city and often causes small explosions with her bombs and everyone worries about it constantly",
    "zhongli remembers old stories",
    "diluc runs the winery outside the city walls at night",
    "venti hums an old song",
]

vocab_words = sorted(set(word for doc in documents for word in doc.split()))
stoi = {w: i for i, w in enumerate(vocab_words)}
PAD_ID = len(stoi)
EOS_ID = len(stoi) + 1
vocab_size = len(stoi) + 2

encoded_docs = [[stoi[w] for w in doc.split()] for doc in documents]
doc_lengths = [len(d) for d in encoded_docs]
print("document lengths:", doc_lengths)

max_len = max(doc_lengths)
padded = torch.full((len(encoded_docs), max_len), PAD_ID)
for i, d in enumerate(encoded_docs):
    padded[i, :len(d)] = torch.tensor(d)

real_tokens = sum(doc_lengths)
total_slots = padded.numel()
pad_fraction = 1 - real_tokens / total_slots
print(f"padded batch shape: {tuple(padded.shape)}")
print(f"real tokens: {real_tokens} / {total_slots} slots -> {pad_fraction:.1%} wasted on padding")

## 2. Concatenate-and-chunk: what real pretraining pipelines actually do

Join every document into one long token stream, separated by an `EOS`
token so the model can tell where one document ends and the next begins,
then slice that stream into fixed-size blocks. No padding, ever — every
position is either real content or a meaningful separator.

In [ ]:
concatenated = []
for d in encoded_docs:
    concatenated.extend(d)
    concatenated.append(EOS_ID)
concatenated = torch.tensor(concatenated)

assert (concatenated == PAD_ID).sum().item() == 0
print(f"concatenated stream length: {len(concatenated)} tokens, 0 padding tokens (verified)")

block_size = 8
num_chunks = (len(concatenated) - 1) // block_size
chunks_x = torch.stack([concatenated[i * block_size:(i + 1) * block_size] for i in range(num_chunks)])
chunks_y = torch.stack([concatenated[i * block_size + 1:(i + 1) * block_size + 1] for i in range(num_chunks)])

print(f"{num_chunks} chunks of shape {tuple(chunks_x.shape)}")

# correctness: within a chunk, y really is x shifted by one position in the underlying stream
for i in range(num_chunks):
    expected_y = concatenated[i * block_size + 1: i * block_size + 1 + block_size]
    assert torch.equal(chunks_y[i], expected_y)
print("Verified: every chunk\'s target is exactly the token stream shifted by one position.")

## 3. Wrapping it in a real `Dataset` + `DataLoader`

In [ ]:
class ChunkedTextDataset(torch.utils.data.Dataset):
    def __init__(self, token_stream, block_size):
        self.token_stream = token_stream
        self.block_size = block_size

    def __len__(self):
        return (len(self.token_stream) - 1) // self.block_size

    def __getitem__(self, idx):
        start = idx * self.block_size
        x = self.token_stream[start:start + self.block_size]
        y = self.token_stream[start + 1:start + 1 + self.block_size]
        return x, y


dataset = ChunkedTextDataset(concatenated, block_size)
loader = torch.utils.data.DataLoader(dataset, batch_size=3, shuffle=True)

batch_x, batch_y = next(iter(loader))
print(f"one batch: x {tuple(batch_x.shape)}, y {tuple(batch_y.shape)}")
assert batch_x.shape == batch_y.shape == (min(3, len(dataset)), block_size)
print(f"DataLoader over {len(dataset)} chunks works as expected.")

## 4. Streaming: never holding the whole corpus in memory at once

A real corpus might not fit in RAM at all. Instead of building one big
`concatenated` tensor up front, a streaming pipeline processes one document
at a time from a generator (standing in for "read the next line from a
multi-gigabyte file on disk"), keeping only a small rolling buffer.
Critically, this must produce the **exact same chunks** as the
materialize-everything approach above — streaming is a memory-efficiency
technique, not a different algorithm.

In [ ]:
def document_stream():
    """Simulates reading documents one at a time from disk - at any moment,
    only the current document (not the whole corpus) needs to be in memory."""
    for doc in documents:
        yield [stoi[w] for w in doc.split()] + [EOS_ID]


class StreamingChunkedDataset(torch.utils.data.IterableDataset):
    def __init__(self, doc_stream_fn, block_size):
        self.doc_stream_fn = doc_stream_fn
        self.block_size = block_size

    def __iter__(self):
        buffer = []
        for doc_tokens in self.doc_stream_fn():
            buffer.extend(doc_tokens)
            while len(buffer) >= self.block_size + 1:
                x = torch.tensor(buffer[:self.block_size])
                y = torch.tensor(buffer[1:self.block_size + 1])
                yield x, y
                buffer = buffer[self.block_size:]
        # any final leftover shorter than block_size + 1 is dropped, same as // above


streaming_dataset = StreamingChunkedDataset(document_stream, block_size)
streamed_chunks = list(streaming_dataset)
print(f"streaming produced {len(streamed_chunks)} chunks (materialized approach had {num_chunks})")

assert len(streamed_chunks) == num_chunks
for i, (sx, sy) in enumerate(streamed_chunks):
    assert torch.equal(sx, chunks_x[i]), f"chunk {i} input mismatch"
    assert torch.equal(sy, chunks_y[i]), f"chunk {i} target mismatch"
print("Confirmed: streaming, one document at a time, produces IDENTICAL chunks to materializing the whole corpus at once.")

## Recap

- Padding every document individually wastes real compute on meaningless
  tokens — measured directly above.
- Concatenating documents with an `EOS` separator and slicing into fixed
  blocks (what GPT-2-style pretraining actually does) uses every position
  for real content, verified to have zero padding.
- A streaming `IterableDataset` processes one document at a time and
  produces byte-for-byte identical chunks to the "load everything into one
  tensor" approach — confirmed by direct comparison — while never needing
  the full corpus in memory. This is exactly what lets Module 30's
  real-sized corpus be processed without needing gigabytes of RAM at once.

Module 22 covers mixed precision training — the other big lever for
fitting a larger training run into limited GPU memory.